# How to use this colab to train, export, load and serve MNIST

## Start the colab custom kernel

* Update the `XM_CELL` and `TPU_PLATFORM` in
 [notebook_custom_kernel_launch.sh](/learning/infra/mira/experimental/orbax_model/python/colab/notebook_custom_kernel_launch.sh) if needed. In partciular, the number of TPUs specified in `TPU_PLATFORM` has to have 8 TPUs which is what this colab assumes.

* Run the command to launch colab custom kernel
```
bash learning/infra/mira/experimental/orbax_model/python/colab/notebook_custom_kernel_launch.sh
```

* Wait until colab custom kernel xm job is up in xm

## Connect to the custom kernel

* Connect to the custom kernel name `${USER}-custom-kernel-${TPU_TOPOLOGY}`

# Create and train a MNIST model

In [ ]:
# @title MNIST model
from flax import linen as nn


class Mnist(nn.Module):
  """A simple CNN model."""

  @nn.compact
  def __call__(self, x):
    x = nn.Conv(features=32, kernel_size=(3, 3))(x)
    x = nn.relu(x)
    x = nn.avg_pool(x, window_shape=(2, 2), strides=(2, 2))
    x = nn.Conv(features=64, kernel_size=(3, 3))(x)
    x = nn.relu(x)
    x = nn.avg_pool(x, window_shape=(2, 2), strides=(2, 2))
    x = x.reshape((x.shape[0], -1))  # flatten
    x = nn.Dense(features=256)(x)
    x = nn.relu(x)
    x = nn.Dense(features=10)(x)
    return x

In [ ]:
# @title Get train and test dataset
import os
import absl.flags as flags
import jax
import jax.numpy as jnp
import tensorflow_datasets as tfds


def get_datasets():
  """Gets MNIST train and test datasets into memory."""
  ds_builder = tfds.builder('mnist')
  ds_builder.download_and_prepare()
  train_ds = tfds.as_numpy(ds_builder.as_dataset(split='train', batch_size=-1))
  test_ds = tfds.as_numpy(ds_builder.as_dataset(split='test', batch_size=-1))
  train_ds['image'] = jnp.float32(train_ds['image']) / 255.0
  test_ds['image'] = jnp.float32(test_ds['image']) / 255.0
  return train_ds, test_ds

In [ ]:
# @test {"timeout": 300}
# @title Train Mnist (code from flax/examples/mnist/train.py)
from absl import logging
from flax.metrics import tensorboard
from flax.training import train_state
import jax as jax
import ml_collections
import numpy as np
import optax


@jax.jit
def apply_model(state, images, labels):
  """Computes gradients, loss and accuracy for a single batch."""

  def loss_fn(params):
    logits = state.apply_fn({'params': params}, images)
    one_hot = jax.nn.one_hot(labels, 10)
    loss = jnp.mean(optax.softmax_cross_entropy(logits=logits, labels=one_hot))
    return loss, logits

  grad_fn = jax.value_and_grad(loss_fn, has_aux=True)
  (loss, logits), grads = grad_fn(state.params)
  accuracy = jnp.mean(jnp.argmax(logits, -1) == labels)
  return grads, loss, accuracy


@jax.jit
def update_model(state, grads):
  return state.apply_gradients(grads=grads)


def create_train_state(rng, config):
  """Creates initial `TrainState`."""
  mnist = Mnist()
  params = mnist.init(rng, jnp.ones([1, 28, 28, 1]))['params']
  tx = optax.sgd(config.learning_rate, config.momentum)
  return train_state.TrainState.create(
      apply_fn=mnist.apply, params=params, tx=tx
  )


def train_epoch(state, train_ds, batch_size, rng):
  """Train for a single epoch."""
  train_ds_size = len(train_ds['image'])
  steps_per_epoch = train_ds_size // batch_size

  perms = jax.random.permutation(rng, len(train_ds['image']))
  perms = perms[: steps_per_epoch * batch_size]  # skip incomplete batch
  perms = perms.reshape((steps_per_epoch, batch_size))

  epoch_loss = []
  epoch_accuracy = []

  for perm in perms:
    batch_images = train_ds['image'][perm, ...]
    batch_labels = train_ds['label'][perm, ...]
    grads, loss, accuracy = apply_model(state, batch_images, batch_labels)
    state = update_model(state, grads)
    epoch_loss.append(loss)
    epoch_accuracy.append(accuracy)
  train_loss = np.mean(epoch_loss)
  train_accuracy = np.mean(epoch_accuracy)
  return state, train_loss, train_accuracy


def train_and_evaluate(
    config: ml_collections.ConfigDict, workdir: str
) -> train_state.TrainState:
  """Execute model training and evaluation loop.

  Args:
    config: Hyperparameter configuration for training and evaluation.
    workdir: Directory where the tensorboard summaries are written to.

  Returns:
    The train state (which includes the `.params`).
  """
  train_ds, test_ds = get_datasets()
  rng = jax.random.key(0)

  summary_writer = tensorboard.SummaryWriter(workdir)
  summary_writer.hparams(dict(config))

  rng, init_rng = jax.random.split(rng)
  state = create_train_state(init_rng, config)

  for epoch in range(1, config.num_epochs + 1):
    rng, input_rng = jax.random.split(rng)
    state, train_loss, train_accuracy = train_epoch(
        state, train_ds, config.batch_size, input_rng
    )
    _, test_loss, test_accuracy = apply_model(
        state, test_ds['image'], test_ds['label']
    )

    logging.warning(
        'epoch:% 3d, train_loss: %.4f, train_accuracy: %.2f, test_loss: %.4f,'
        ' test_accuracy: %.2f'
        % (
            epoch,
            train_loss,
            train_accuracy * 100,
            test_loss,
            test_accuracy * 100,
        )
    )

    summary_writer.scalar('train_loss', train_loss, epoch)
    summary_writer.scalar('train_accuracy', train_accuracy, epoch)
    summary_writer.scalar('test_loss', test_loss, epoch)
    summary_writer.scalar('test_accuracy', test_accuracy, epoch)

  summary_writer.flush()
  return state


def get_config():
  """Get the default hyperparameter configuration."""
  config = ml_collections.ConfigDict()
  config.learning_rate = 0.1
  config.momentum = 0.9
  config.batch_size = 128
  config.num_epochs = 3
  return config


import os
import shutil
import tempfile


def create_temp_subdir(subdir_name, override=False):
  temp_dir = tempfile.gettempdir()
  subdir_path = os.path.join(temp_dir, subdir_name)

  try:
    if override and os.path.exists(subdir_path):
      shutil.rmtree(subdir_path)  # Remove the existing subdirectory
    os.makedirs(subdir_path, exist_ok=True)
    return subdir_path
  except OSError as e:
    return None


workdir = create_temp_subdir('mnist')
state = train_and_evaluate(get_config(), workdir)

# Export the model

In [ ]:
# @title Export model
from collections.abc import Mapping
import os
from etils import epath
import flax.linen as nn
import frozendict
import jax
from jax.experimental import mesh_utils
import jax.numpy as jnp
import jax.sharding as jsharding
import orbax.checkpoint as ocp
from orbax.export import constants
from orbax.export import export_manager
from orbax.export import jax_module
from orbax.export import obm_configs
from orbax.export import serving_config as osc
import tensorflow as tf


# This is the pre-processing function
# Not necessary to decorate with tf.function here (unless signature change is
# required). The export function will do it.
def process_image(image: tf.Tensor) -> tf.Tensor:
  # x is a uint8 tensor of shape (b, length, width, 3).
  gray_image = tf.image.rgb_to_grayscale(image)
  resized_image = tf.image.resize(gray_image, [28, 28]) / 255.0
  return resized_image


# This is the post-processing function
# No need to decorate with tf.function here. The export function will do it.
def select_digit(x: tf.Tensor) -> Mapping[str, tf.Tensor]:
  return {'predicted_label': tf.math.argmax(x, axis=1)}


class MnistExporter:
  """Exports Mnist model."""

  def __init__(
      self,
      params,
      output_path: epath.Path = epath.Path('/tmp/mnist'),
      ici_mesh: Mapping[str, int] = frozendict.frozendict({
          'data': 1,
          'model': 1,
      }),
  ):
    """Initializes the exporter.

    Args:
      params: Params for the mnist.
      output_path: The path to exported model.
      ici_mesh: The ICI mesh for sharding.
    """
    self._params = {'params': params}
    self._create_output_dir(epath.Path(output_path))
    self._output_path = output_path
    self._ckpt_subdir = 'ckpt'
    self._ici_mesh = ici_mesh

  def _create_output_dir(self, output_path: epath.Path):
    """Creates the output directory."""
    if output_path.exists():
      output_path.rmtree()
    epath.Path(output_path).mkdir(parents=True, exist_ok=False)

  def _create_params_sharding_spec(
      self, mesh: jax.sharding.Mesh, p: jax.ShapeDtypeStruct
  ) -> jax.sharding.NamedSharding:
    specs = (
        (None,) * (len(p.shape) - 1) + ('model',) if len(p.shape) > 0 else ()
    )
    return jax.sharding.NamedSharding(mesh, jsharding.PartitionSpec(*specs))

  def _create_mesh(self) -> jax.sharding.Mesh:
    devices = mesh_utils.create_device_mesh(tuple(self._ici_mesh.values()))
    return jax.sharding.Mesh(devices, tuple(self._ici_mesh.keys()))

  def _save_ckpt(self):
    """Saves the checkpoint."""
    ckpter = ocp.Checkpointer(ocp.StandardCheckpointHandler())
    ckpter.save(self._output_path / self._ckpt_subdir, self._params)

  def export(self):
    """Exports a mnist jax model with untrained weights."""
    model = Mnist()

    mesh = self._create_mesh()

    params_sharding_spec = jax.tree_util.tree_map(
        lambda p: self._create_params_sharding_spec(mesh, p), self._params
    )
    params_arg_spec = jax.tree_util.tree_map(
        lambda p: jax.ShapeDtypeStruct(p.shape, p.dtype), self._params
    )
    inputs_sharding_spec = jax.sharding.NamedSharding(
        mesh, jsharding.PartitionSpec('data', 'model', None, None)
    )

    model_apply_fn = jax.jit(
        model.apply,
        in_shardings=(
            params_sharding_spec,
            inputs_sharding_spec,
        ),
        out_shardings=jax.sharding.NamedSharding(
            mesh,
            jsharding.PartitionSpec(None),
        ),
    )

    model_function_name = 'mnist_forward_fn'
    weights_name = 'params'
    self._save_ckpt()

    # This is the overall input arg spec. For this example, it is the input arg
    # spec for TF Preprocessing function.
    input_args_spec = [tf.TensorSpec((None, None, None, 3), tf.uint8)]

    em = export_manager.ExportManager(
        jax_module.JaxModule(
            params=params_arg_spec,
            apply_fn={model_function_name: model_apply_fn},
            input_polymorphic_shape={model_function_name: f'b, 28, 28, 1'},
            export_version=constants.ExportModelType.ORBAX_MODEL,
            jax2obm_kwargs={
                constants.CHECKPOINT_PATH: os.fspath(self._ckpt_subdir),
                constants.WEIGHTS_NAME: weights_name,
                # Honor `inputs_sharding_spec`
                constants.POLYMORPHIC_CONSTRAINTS: [
                    f'mod(b, {self._ici_mesh["data"]}) == 0'
                ],
            },
        ),
        serving_configs=[
            osc.ServingConfig(
                signature_key='serving_default',
                input_signature=input_args_spec,
                tf_preprocessor=process_image,
                tf_postprocessor=select_digit,
                obm_export_options=obm_configs.ObmExportOptions(
                    # Use the following to enable batching
                    batch_options=obm_configs.BatchOptions(
                        batch_component=obm_configs.BatchComponent.MODEL_FUNCTION,
                        max_batch_size=128,
                        batch_timeout_micros=1000,
                        # Every batch size in the list needs to be divisible by the number
                        # of devices along the sharding dimension (`data`).
                        allowed_batch_sizes=(4, 8, 16, 32),
                        num_batch_threads=4,
                        max_enqueued_batches=250,
                    ),
                ),
            )
        ],
    )

    em.save(os.fspath(self._output_path))


exporter = MnistExporter(
    state.params,
    output_path=epath.Path('/tmp/mnist'),
    ici_mesh=frozendict.frozendict({'data': 4, 'model': 2}),
)
exporter.export()

# Load the model and run inference

In [ ]:
# @title Load model
from .third_party.neptune.runtime.python import orbax_model_runner

flags.FLAGS.orbax_model_runner_platform = 'tpu'

runner = orbax_model_runner.ModelRunner('mnist', '/tmp/mnist')
print(
    f'Stable HLO will run on {runner.ifrt_platform_name()} platform with'
    f' {runner.ifrt_device_count()} devices.'
)
if runner.ifrt_platform_name() != 'tpu':
  raise RuntimeError('Expect the Orbax Runner to use TPU.')

In [ ]:
# @test {"timeout": 90}
# @title Run model on test dataset
import itertools

ds_builder = tfds.builder('mnist')
ds_builder.download_and_prepare()
dataset = ds_builder.as_dataset(split='test', shuffle_files=True)

signature = runner.signature('serving_default')
predicted_labels = []

# Trigger concating and large batch splitting.
for chuck in itertools.batched(dataset, 17):
  images = tf.stack([x['image'] for x in chuck])
  # Convert to RGB because preprocessing requires RGB
  rbg_images = tf.image.grayscale_to_rgb(images)
  predicted_labels.extend(
      signature.run_with_dict_return(image=rbg_images)['predicted_label']
  )

actual_labels = [x['label'].numpy() for x in dataset]

accuracy = np.mean(np.stack(predicted_labels) == np.stack(actual_labels))
print(f'Accuracy: {accuracy}')
assert accuracy > 0.9

In [ ]:
# @test {"skip": true}
# @title Draw an image with mouse
from base64 import b64decode
from google.colab.output import eval_js
from IPython.display import HTML, display
import tensorflow as tf

canvas_html = """
<canvas width=%d height=%d></canvas>
<button>Finish</button>
<script>
var canvas = document.querySelector('canvas')
var ctx = canvas.getContext('2d')
ctx.lineWidth = %d
ctx.strokeStyle = 'rgba(255,255,0,1)';
ctx.fillStyle = 'rgba(0,0,0,0.9)';
ctx.fillRect(0, 0, ctx.canvas.width, ctx.canvas.height)
var button = document.querySelector('button')
var mouse = {x: 0, y: 0}
canvas.addEventListener('mousemove', function(e) {
  mouse.x = e.pageX - this.offsetLeft
  mouse.y = e.pageY - this.offsetTop
})
canvas.onmousedown = ()=>{
  ctx.beginPath()
  ctx.moveTo(mouse.x, mouse.y)
  canvas.addEventListener('mousemove', onPaint)
}
canvas.onmouseup = ()=>{
  canvas.removeEventListener('mousemove', onPaint)
}
var onPaint = ()=>{
  ctx.lineTo(mouse.x, mouse.y)
  ctx.stroke()
}
var data = new Promise(resolve=>{
  button.onclick = ()=>{
    resolve(canvas.toDataURL('image/png'))
  }
})
</script>
"""


def draw(w=300, h=300, line_width=20):
  display(HTML(canvas_html % (w, h, line_width)))
  data = eval_js("data")
  binary = b64decode(data.split(",")[1])
  return binary


image_data = draw()
original_image = tf.image.decode_image(image_data, 3)

In [ ]:
# @test {"skip": true}
# @title Run inference on the mouse-drawn picture
from matplotlib import pyplot as plt
import numpy as np
from PIL import Image


def plot_image(image, title=""):
  """Plots images from image tensors.

  Args:
    image: 3D image tensor. [height, width, channels].
    title: Title to display in the plot.
  """
  image = np.asarray(image)
  image = tf.clip_by_value(image, 0, 255)
  image = Image.fromarray(tf.cast(image, tf.uint8).numpy())
  plt.imshow(image)
  plt.axis("off")
  plt.title(title)


def _create_model_inputs(image, batch_size):
  inputs = [np.asarray(image)] * batch_size
  batched_inputs = np.stack(inputs, axis=0)
  return batched_inputs


predicted_labels = runner.signature("serving_default").run_with_dict_return(
    image=_create_model_inputs(original_image, batch_size=100)
)["predicted_label"]

assert len(set(predicted_labels)) == 1

plot_image(original_image, title=f"predicted_label={predicted_labels[0]}")